# Deep BSDE: Solving Ergodic BSDEs with Neural Networks

The **Deep BSDE** approach parameterises the unknown functions with neural
networks and trains them by minimising a loss derived from the BSDE equation.

For ergodic BSDEs the `ErgodicDeepBSDESolver` implements three strategies:

| Strategy | Loss | Notes |
|---|---|---|
| `temporal_difference` | TD error along simulated paths | Fast, simulation-based |
| `direct_ergodic` | PDE residual under stationary measure | Meshless |
| `long_horizon` | Runs standard deep-BSDE at large $T$, extracts $\lambda$ | High variance |

In this notebook we use the **`direct_ergodic`** strategy, which minimises

$$
\mathcal{L}(v_\theta, \lambda) = \mathbb{E}_{X\sim\pi}\bigl[
  |\mathcal{L}v_\theta(X) + f(X, v_\theta(X), z_\theta(X)) - \lambda|^2
\bigr] + \alpha\,|v_\theta(x_0)|^2
$$

where $\pi$ is the stationary distribution and the second term normalises $v$.


In [ ]:
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline

import torch
print("PyTorch version:", torch.__version__)

from ebsde.forward.ou_process import OrnsteinUhlenbeck
from ebsde.bsde.ergodic import ErgodicBSDE
from ebsde.solvers.ergodic_deep import ErgodicDeepBSDESolver
from ebsde.solvers.ergodic_pde import ErgodicPDESolver
from data.synthetic import ergodic_ou_quadratic_analytical


## Network Architecture

The solver maintains two networks:
- **$v_\theta$** — maps $x \in \mathbb{R}^d$ to $v(x) \in \mathbb{R}$
  (a 3-hidden-layer MLP with Tanh activations).
- **$z_\theta$** — maps $x$ to $z(x) = \sigma(x)\nabla_x v(x)$
  (a separate 2-hidden-layer MLP).

$\lambda$ is a **trainable scalar parameter**, jointly optimised with Adam.

During training the state samples $X \sim \mathcal{N}(0, \sigma^2/(2\kappa))$
are drawn from the OU stationary distribution at each step.


In [ ]:
import os
# --- Setup ----------------------------------------------------------
ou       = OrnsteinUhlenbeck(kappa=1.0, theta=0.0, sigma=1.0)
driver_q = lambda x, y, z: x**2 - 0.5 * np.sum(np.asarray(z, dtype=float)**2, axis=-1)

ebsde = ErgodicBSDE(forward=ou, driver=driver_q)

# --- Train ----------------------------------------------------------
deep_solver = ErgodicDeepBSDESolver(
    ebsde,
    strategy='direct_ergodic',
    n_epochs=500,
    n_samples=512,
    rng_seed=42,
    torch_seed=42,
)
deep_sol = deep_solver.solve()

lambda_deep = deep_sol['lambda_ergodic']
loss_hist   = deep_sol['training_loss']
lam_hist    = deep_sol['lambda_history']

print(f"lambda (deep)  = {lambda_deep:.6f}")

# --- Training loss curve -------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

epochs = np.arange(1, len(loss_hist) + 1)
ax1.semilogy(epochs, loss_hist, 'C0', lw=1.2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training loss (log scale)')

ax2.plot(np.arange(1, len(lam_hist)+1), lam_hist, 'C1', lw=1.5)
ax2.axhline(lambda_deep, color='k', lw=0.5, ls='--', label=f'final λ={lambda_deep:.4f}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('λ estimate')
ax2.set_title('Ergodic constant during training')
ax2.legend()

fig.tight_layout()
os.makedirs('notebooks/figures', exist_ok=True)
fig.savefig('notebooks/figures/04_training_loss.png', dpi=120)
plt.close(fig)
print("Figure saved → notebooks/figures/04_training_loss.png")


## Comparison: Deep vs Classical Solvers

We compare the ergodic constant $\lambda$ obtained by the deep solver against
the PDE solver and the exact closed-form result.


In [ ]:
# --- Classical PDE solver -------------------------------------------
pde_sol     = ErgodicPDESolver(ebsde, n_x=300).solve()
lambda_pde  = pde_sol['lambda_ergodic']

# --- Exact ----------------------------------------------------------
exact       = ergodic_ou_quadratic_analytical(
    kappa=1.0, sigma=1.0, alpha=1.0, beta=0.0, gamma=1.0
)
lambda_exact = exact['lambda_ergodic']

# --- Comparison table -----------------------------------------------
print("=" * 55)
print(f"{'Method':<30} {'lambda':>10}  {'|Error|':>10}")
print("-" * 55)
print(f"{'Exact (Cole-Hopf)':<30} {lambda_exact:>10.6f}  {'—':>10}")
print(f"{'ErgodicPDESolver (n_x=300)':<30} {lambda_pde:>10.6f}  {abs(lambda_pde-lambda_exact):>10.2e}")
print(f"{'ErgodicDeepBSDESolver':<30} {lambda_deep:>10.6f}  {abs(lambda_deep-lambda_exact):>10.2e}")
print("=" * 55)
